# Stage 3 -- Model-Ready: Daily Full Moments

## Input
`Data/Data_Collection/Final/Stage_2/agg_market_daily_full_moments.parquet` -- aggregated daily table with five cap-weighted cross-sectional moments per stock factor plus macro daily factors, keyed on `date`

## Purpose
Applies expanding-window z-standardisation to the aggregated daily full moments table to produce the final model-ready dataset. The z-scoring approach is identical to the daily means notebook: all continuous features are standardised using only historical data up to t-1, preventing any look-ahead. Binary, bounded, and calendar features are left in their raw form.

---

## Pipeline

### Step 1: Load
The Stage 2 aggregated full moments table is loaded and sorted by date. Column counts are reported separately for stock moment columns (identified by `_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread` suffixes) and macro/other columns.

### Step 2: Identify Columns to Z-Score vs Skip
Columns are split into two groups:

**Skipped (not z-scored):**
- `date` and `target_daily_return` -- meta columns
- **Binary/indicator features** (from Panel C macro): `is_monday`, `is_friday`, `is_quarter_end`, `is_turn_of_month`, `is_opex_week`, `vix_above_20`, `vix_above_30`, `curve_inverted_2y10y`, `curve_inverted_3m10y`, `credit_stress`
- **Categorical/ordinal features:** `day_of_week` (0--4), `month_of_year` (1--12), `trading_days_to_month_end` (0--22)

**Z-scored:** all remaining columns -- stock moment columns (cwmean, cwstd, cwskew, cwkurt, spread for each factor) plus continuous macro features.

**Pre-z-score drop:** `dlyreti_spread` is dropped before z-scoring. Dividends are rare discrete daily events, so the cross-sectional spread of `dlyreti` is near-constant zero for years, causing the expanding standard deviation to remain near zero and producing undefined z-scores until approximately 2013. It is removed rather than left as a sparse NaN column.

### Step 3: Expanding-Window Z-Standardisation
Applied to all `zscore_cols` using the formula:

`z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}`

- **`shift(1)` applied to both expanding mean and std** -- the current observation is excluded from its own standardisation, preventing look-ahead
- **Minimum 252 trading days (~1 year)** before the first valid z-score is produced
- Computed in one vectorised pass over all z-scored columns using pandas `expanding().mean()` and `expanding().std()` followed by `shift(1)`
- Expanding mean and std arrays are deleted immediately after use to free memory
- Any resulting ±inf values (from σ = 0 periods on near-constant factors) are replaced with NaN

### Step 4: Drop Warmup Rows
The first `MIN_WINDOW + 1` rows (253 rows: 252-day expanding warmup + 1 for the shift) are dropped. If any z-scored columns still contain NaN after this trim (from near-constant factors with late-starting variance), additional rows are trimmed until all NaN are eliminated. The start date is then aligned to 2007-11-30 to match the combined tables.

### Step 5: Validate
- **NaN check:** zero NaN expected in all feature columns after warmup trim; any remaining are listed
- **Infinite value check:** confirms no ±inf remain after the replacement in Step 3
- **Zero-variance check:** identifies any columns that are all-NaN or constant after z-scoring
- **Target integrity:** mean (~0.0005), std (~0.012), min, max, NaN count -- confirms target was not z-scored
- **Z-score distribution check:** for one sample base factor, all five moment columns are shown with their post-z-score mean, std, min, max (expect mean ≈ 0, std ≈ 1); three macro features also shown
- **Binary feature check:** confirms skipped binary columns still contain only 0/1 values
- **No duplicate dates**

### Step 6: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **Identical z-scoring approach to the daily means notebook** -- same `shift(1)`, same `MIN_WINDOW = 252`, same skip list for binary/calendar features.
- **`dlyreti_spread` dropped** because the p90-p10 spread of a near-zero column produces an undefined expanding z-score for the first several years of the sample. This is specific to the full moments table; the cwmean of `dlyreti` in the means table does not have this problem.
- **Start date aligned to 2007-11-30** after warmup trim to match the combined (daily + monthly) tables, enabling consistent date ranges across all Stage 3 outputs.
- **All five moments z-scored together** in a single vectorised pass -- there is no distinction in z-scoring treatment between cwmean, cwstd, cwskew, cwkurt, and spread columns.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_full_moments.parquet` -- keyed on `date`, all continuous features expanding-window z-standardised using only past data, binary/calendar features in raw form, `target_daily_return` in raw returns

In [1]:
# %% [markdown]
# # Stage 3 — Model-Ready: Daily Full Moments
#
# Applies expanding-window z-standardisation to the aggregated daily full
# moments table (cwmean, cwstd, cwskew, cwkurt, spread per stock factor
# + macro factors), producing the final model-ready dataset.
#
# Z-scoring approach (identical to daily means):
#   z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
#   - Uses ONLY data up to t-1 (shift(1) ensures no look-ahead)
#   - Minimum 252 trading days (~1 year) before first valid z-score
#   - Binary/bounded/calendar features are NOT z-scored
#   - Target variable is NOT z-scored (stays in raw returns)
#
# Input:  Stage_2/agg_market_daily_full_moments.parquet
# Output: Stage_3_Model_Ready/model_market_daily_full_moments.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

IN_PATH = Path('../../../../Data/Data_Collection/Final/Stage_2/agg_market_daily_full_moments.parquet')
OUT_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD")
print("=" * 90)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"\n  Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Column type breakdown
moment_suffixes = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']
moment_cols = [c for c in df.columns if any(c.endswith(s) for s in moment_suffixes)]
non_moment_cols = [c for c in df.columns if c not in moment_cols and c not in ['date', 'target_daily_return']]
print(f"  Stock moment columns: {len(moment_cols)}")
print(f"  Macro/other columns: {len(non_moment_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP")
print("=" * 90)

# Meta columns — never z-scored
meta_cols = ['date', 'target_daily_return']

# Binary/bounded features (from Panel C macro)
skip_binary = [
    'is_monday', 'is_friday', 'is_quarter_end',
    'is_turn_of_month', 'is_opex_week',
    'vix_above_20', 'vix_above_30',
    'curve_inverted_2y10y', 'curve_inverted_3m10y',
    'credit_stress',
]

# Categorical/ordinal features
skip_categorical = [
    'day_of_week',               # 0-4
    'month_of_year',             # 1-12
    'trading_days_to_month_end', # 0-22
]

# Combine (only those present in data)
all_skip = set(meta_cols + skip_binary + skip_categorical)
all_skip = {c for c in all_skip if c in df.columns}

# Everything else gets z-scored (moment columns + macro continuous features)
zscore_cols = [c for c in df.columns if c not in all_skip]

print(f"\n  Total columns: {df.shape[1]}")
print(f"  Columns to z-score: {len(zscore_cols)}")
print(f"  Columns to skip: {len(all_skip)}")

print(f"\n  Skipped columns:")
print(f"    Meta:        {[c for c in meta_cols if c in df.columns]}")
print(f"    Binary:      {[c for c in skip_binary if c in df.columns]}")
print(f"    Categorical: {[c for c in skip_categorical if c in df.columns]}")

# Breakdown of z-scored columns by type
zscore_moment = [c for c in zscore_cols if any(c.endswith(s) for s in moment_suffixes)]
zscore_macro = [c for c in zscore_cols if c not in zscore_moment]
print(f"\n  Z-scored breakdown:")
print(f"    Stock moment columns: {len(zscore_moment)}")
print(f"    Macro/other columns:  {len(zscore_macro)}")



# Drop dlyreti_spread — near-constant zero (dividends are rare daily events),
# expanding σ ≈ 0 for years, making z-scores undefined until 2013
if 'dlyreti_spread' in df.columns:
    df = df.drop(columns=['dlyreti_spread'])
    zscore_cols = [c for c in zscore_cols if c != 'dlyreti_spread']
    print(f"\n  Dropped dlyreti_spread (near-zero for years, undefined z-score)")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: EXPANDING-WINDOW Z-STANDARDISATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: EXPANDING-WINDOW Z-STANDARDISATION")
print("=" * 90)

MIN_WINDOW = 252

t0 = time.time()

print(f"\n  Z-scoring {len(zscore_cols)} columns with expanding window (min {MIN_WINDOW} days)...")
print(f"  Formula: z_t = (x_t - μ_{{1:t-1}}) / σ_{{1:t-1}}")
print(f"  shift(1) ensures NO look-ahead — current day excluded from mean/std\n")

# Expanding mean and std, shifted by 1 to exclude current observation
expanding_mean = df[zscore_cols].expanding(min_periods=MIN_WINDOW).mean().shift(1)
expanding_std = df[zscore_cols].expanding(min_periods=MIN_WINDOW).std().shift(1)

# Apply z-score
df[zscore_cols] = (df[zscore_cols] - expanding_mean) / expanding_std

# Free memory
del expanding_mean, expanding_std
import gc; gc.collect()

elapsed = time.time() - t0
print(f"  Z-scoring completed in {elapsed:.1f}s")

# Replace any inf/-inf from division by zero (σ = 0 for constant factors)
inf_before = np.isinf(df[zscore_cols]).sum().sum()
if inf_before > 0:
    df[zscore_cols] = df[zscore_cols].replace([np.inf, -np.inf], np.nan)
    print(f"  ⚠ Replaced {inf_before} infinite values with NaN (from σ = 0 periods)")
else:
    print(f"  ✓ No infinite values produced")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: DROP WARMUP ROWS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: DROP WARMUP ROWS")
print("=" * 90)

# Drop first MIN_WINDOW + 1 rows (expanding warmup + shift)
warmup_needed = MIN_WINDOW + 1

pre_drop = len(df)
warmup_date = df.iloc[warmup_needed - 1]['date']
df = df.iloc[warmup_needed:].reset_index(drop=True)

print(f"\n  Dropped first {warmup_needed} rows (warmup period)")
print(f"  Warmup end date: {warmup_date.date()}")

# Trim any additional NaN from near-constant factors in early expanding window
nan_rows = df[df[zscore_cols].isna().any(axis=1)]
if len(nan_rows) > 0:
    last_nan_row = max(nan_rows.index)
    pre = len(df)
    df = df.iloc[last_nan_row + 1:].reset_index(drop=True)
    print(f"  Trimmed {pre - len(df)} additional rows to remove early z-score NaN")
else:
    print(f"  ✓ No additional NaN rows to trim")


# Align start date with combined tables (2007-11-30)
df = df[df['date'] >= '2007-11-30'].reset_index(drop=True)
print(f"  Aligned to combined start date: {len(df):,} rows from {df['date'].min().date()}")


print(f"\n  Rows: {pre_drop:,} → {len(df):,}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATE")
print("=" * 90)

# 5a. NaN check
feature_cols = [c for c in df.columns if c not in ['date', 'target_daily_return']]
feature_nan = df[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(20).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"\n  ✓ Zero NaN in features")

# 5b. Infinite values
inf_count = 0
inf_cols = []
for c in zscore_cols:
    if c in df.columns:
        n_inf = np.isinf(df[c]).sum()
        if n_inf > 0:
            inf_count += n_inf
            inf_cols.append((c, n_inf))

if inf_count > 0:
    print(f"\n  ⚠ Infinite values: {inf_count}")
    for c, n in inf_cols[:15]:
        print(f"    {c}: {n}")
else:
    print(f"  ✓ Zero infinite values")

# 5c. Zero-variance columns
zero_var_cols = []
for c in zscore_cols:
    if c in df.columns:
        if df[c].isna().all():
            zero_var_cols.append(c)
        elif pd.notna(df[c].std()) and float(df[c].std()) == 0:
            zero_var_cols.append(c)

if zero_var_cols:
    print(f"\n  ⚠ Zero-variance after z-score ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        print(f"    {c}")
else:
    print(f"  ✓ No zero-variance columns")

# 5d. Target untouched
print(f"\n  Target statistics (should be raw returns, NOT z-scored):")
print(f"    Mean:  {df['target_daily_return'].mean():.6f} (expect ~0.0005)")
print(f"    Std:   {df['target_daily_return'].std():.6f} (expect ~0.012)")
print(f"    Min:   {df['target_daily_return'].min():.6f}")
print(f"    Max:   {df['target_daily_return'].max():.6f}")
print(f"    NaN:   {df['target_daily_return'].isna().sum()}")

# 5e. Z-score distribution check (sample from each moment type)
print(f"\n  Z-score distribution check (one sample per moment type):")
print(f"  {'Column':<45s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("  " + "-" * 75)

# Pick one factor and show all 5 moments
sample_base = None
for c in df.columns:
    if c.endswith('_cwmean') and not c.startswith('stock_'):
        sample_base = c.replace('_cwmean', '')
        break

if sample_base:
    for suffix in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        col = f'{sample_base}{suffix}'
        if col in df.columns:
            vals = df[col].dropna()
            if len(vals) > 0:
                print(f"  {col:<45s} {vals.mean():>8.3f} {vals.std():>8.3f} "
                      f"{vals.min():>8.2f} {vals.max():>8.2f}")

# Also show a few macro features
macro_samples = [c for c in zscore_macro if c in df.columns][:3]
for c in macro_samples:
    vals = df[c].dropna()
    if len(vals) > 0:
        print(f"  {c:<45s} {vals.mean():>8.3f} {vals.std():>8.3f} "
              f"{vals.min():>8.2f} {vals.max():>8.2f}")

# 5f. Binary features untouched
binary_in_data = [c for c in skip_binary if c in df.columns]
if binary_in_data:
    print(f"\n  Binary features (should still be 0/1):")
    for c in binary_in_data[:5]:
        unique = sorted(df[c].dropna().unique())
        print(f"    {c}: unique values = {unique}")

# 5g. No duplicate dates
n_dupes = df['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: SAVE")
print("=" * 90)

df = df.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'model_market_daily_full_moments.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY DAILY FULL MOMENTS COMPLETE")
print("=" * 90)

n_zscored = len(zscore_cols)
n_skipped = len([c for c in all_skip if c in df.columns])

print(f"""
  Input:  agg_market_daily_full_moments.parquet (Stage 2)
  Output: model_market_daily_full_moments.parquet (Stage 3)

  Z-standardisation:
    Method:     Expanding window, shift(1), min {MIN_WINDOW} days
    Z-scored:   {n_zscored} features ({len(zscore_moment)} stock moments + {len(zscore_macro)} macro)
    Skipped:    {n_skipped} features (binary/calendar/target)
    Warmup:     {warmup_needed}+ rows dropped

  Result:
    Rows:       {df.shape[0]:,} trading days
    Columns:    {df.shape[1]}
    Dates:      {df['date'].min().date()} → {df['date'].max().date()}
    NaN:        {feature_nan_total} features + {df['target_daily_return'].isna().sum()} target

  Saved: {out_path}
""")

STEP 1: LOAD

  Loaded: 4,605 rows × 1148 columns
  Date range: 2006-09-13 → 2024-12-30
  Stock moment columns: 948
  Macro/other columns: 198

STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP

  Total columns: 1148
  Columns to z-score: 1133
  Columns to skip: 15

  Skipped columns:
    Meta:        ['date', 'target_daily_return']
    Binary:      ['is_monday', 'is_friday', 'is_quarter_end', 'is_turn_of_month', 'is_opex_week', 'vix_above_20', 'vix_above_30', 'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress']
    Categorical: ['day_of_week', 'month_of_year', 'trading_days_to_month_end']

  Z-scored breakdown:
    Stock moment columns: 948
    Macro/other columns:  185

  Dropped dlyreti_spread (near-zero for years, undefined z-score)

STEP 3: EXPANDING-WINDOW Z-STANDARDISATION

  Z-scoring 1132 columns with expanding window (min 252 days)...
  Formula: z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
  shift(1) ensures NO look-ahead — current day excluded from mean/std

  Z-scoring complet